In [1]:
# Run this FIRST before any model loading
import torch, gc

# Clear everything
try: del model
except: pass
try: del finetuned_model
except: pass
try: del baseline_model
except: pass
try: del base
except: pass

gc.collect()
torch.cuda.empty_cache()

# Check what's free
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM free: {total - used:.1f} / {total:.1f} GB")

VRAM free: 8.6 / 8.6 GB


# RuralMed Vision — Evaluation Suite
**Test set evaluation, baseline comparison, AUC-ROC, calibration**

This notebook produces the scientific results needed for the paper:
1. Held-out test set (never seen during training)
2. Zero-shot baseline (Qwen2.5-VL without fine-tuning)
3. Fine-tuned model results
4. Per-class AUC-ROC
5. Confusion matrix
6. Confidence calibration (ECE)

In [2]:
# Cell 1 -- Setup
import os, sys, json, re, yaml, torch, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
from PIL import Image
from collections import Counter
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize
warnings.filterwarnings("ignore")

ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)

with open(os.path.join(ROOT, "configs", "config.yaml")) as f:
    cfg = yaml.safe_load(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB")

CLASSES = ["mel","nv","bcc","akiec","bkl","df","vasc"]
CLASS_NAMES = ["Melanoma","Nevi","Basal Cell Ca.","Actinic Keratosis",
               "Benign Keratosis","Dermatofibroma","Vascular"]
SEVERITY_MAP = {"mel":"HIGH","bcc":"HIGH","akiec":"MEDIUM","vasc":"MEDIUM",
                "bkl":"LOW","df":"LOW","nv":"LOW"}

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU | 8.6 GB


In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(ROOT, "data", "HAM10000_metadata.csv"), nrows=1, engine="python")
print(df)

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(os.path.join(ROOT, "data", "HAM10000_metadata.csv"))
image_dirs = cfg["data"]["ham10000_images"]

def find_image(image_id):
    for d in image_dirs:
        for ext in [".jpg", ".jpeg"]:
            p = os.path.join(d, f"{image_id}{ext}")
            if os.path.isfile(p): return p
    return None

sample = df.head(10)
sample["image_path"] = sample["image_id"].map(find_image)
print(sample[["image_id", "image_path"]])

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(os.path.join(ROOT, "data", "HAM10000_metadata.csv"))
image_dirs = cfg["data"]["ham10000_images"]

def find_image(image_id):
    for d in image_dirs:
        for ext in [".jpg", ".jpeg"]:
            p = os.path.join(d, f"{image_id}{ext}")
            if os.path.isfile(p): return p
    return None

df["image_path"] = df["image_id"].map(find_image)
df = df[df["image_path"].notna()].copy()
print(f"Samples with images: {len(df)}")

_, test_df = train_test_split(
    df, test_size=0.30, random_state=99, stratify=df["dx"]
)
print(f"Test set size: {len(test_df)}")
print(test_df["dx"].value_counts())

In [ ]:
# Cell 4 -- Load fine-tuned model (4-bit, no device_map)
import torch, gc
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

torch.cuda.empty_cache(); gc.collect()
used  = torch.cuda.memory_allocated(0)/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"VRAM before load: {used:.1f}/{total:.1f} GB")

ADAPTER_PATH = cfg["model"]["adapter_path"]
processor = AutoProcessor.from_pretrained(ADAPTER_PATH)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # float16 not bfloat16
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForImageTextToText.from_pretrained(
    cfg["model"]["name"],
    quantization_config=bnb,
    torch_dtype=torch.float16,
    # NO device_map here
)
finetuned_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
finetuned_model.eval()

used = torch.cuda.memory_allocated(0)/1e9
print(f"Fine-tuned model ready | VRAM: {used:.1f}/{total:.1f} GB")

In [ ]:
# Cell 5 -- Inference function + SYMPTOM_LOOKUP
import json as jlib, re
from PIL import Image

EVAL_SYSTEM = """You are a medical image classifier. Given a skin lesion image and symptoms,
classify the condition. Respond with ONLY a JSON object:
{
  "condition": "<condition name>",
  "severity": "<LOW, MEDIUM, or HIGH>",
  "dx_code": "<one of: mel, nv, bcc, akiec, bkl, df, vasc>"
}"""

SYMPTOM_LOOKUP = {
    "mel":   "Dark irregular mole, growing, uneven edges.",
    "nv":    "Symmetric brown mole, stable for years.",
    "bcc":   "Pearly bump, slow growing, bleeds when scratched.",
    "akiec": "Rough scaly patch, sun-exposed area, mild itching.",
    "bkl":   "Waxy brown warty lesion, longstanding.",
    "df":    "Small firm nodule, slightly itchy.",
    "vasc":  "Bright red spot, blanches on pressure.",
}

def run_inference(model, proc, image_path, symptom_text, image_size=128):
    image = Image.open(image_path).convert("RGB").resize((image_size, image_size))
    messages = [
        {"role": "system", "content": [{"type": "text", "text": EVAL_SYSTEM}]},
        {"role": "user",   "content": [{"type": "image", "image": image},
                                       {"type": "text",  "text": f"Symptoms: {symptom_text}"}]},
    ]
    text   = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=text, images=[image], return_tensors="pt").to(
        next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False,
                             pad_token_id=proc.tokenizer.eos_token_id)
    raw = proc.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return jlib.loads(re.sub(r"```json|```","",raw).strip())
    except:
        sev = next((s for s in ["HIGH","MEDIUM","LOW"] if s in raw.upper()), "UNKNOWN")
        return {"condition":"parse_error","severity":sev,"dx_code":"nv"}

print("Inference function ready.")

In [ ]:
# Cell 6 -- Fine-tuned model evaluation
# Run this BEFORE loading the baseline -- keeps only one model in VRAM at a time
from tqdm.notebook import tqdm

MAX_EVAL    = min(200, len(test_df))   # increase to len(test_df) for paper results
test_sample = test_df.sample(n=MAX_EVAL, random_state=42).reset_index(drop=True)

ft_results = []
print(f"Evaluating fine-tuned model on {MAX_EVAL} samples...")
print("This takes ~1-2 hours for 200 samples. Leave it running.\n")

for _, row in tqdm(test_sample.iterrows(), total=MAX_EVAL):
    symptom = SYMPTOM_LOOKUP.get(row["dx"], "Skin lesion noted.")
    result  = run_inference(finetuned_model, processor, row["image_path"], symptom)
    ft_results.append({
        "true_dx":        row["dx"],
        "true_severity":  SEVERITY_MAP[row["dx"]],
        "pred_dx":        result.get("dx_code", "nv"),
        "pred_severity":  result.get("severity","UNKNOWN").upper(),
        "pred_condition": result.get("condition",""),
    })

sev_correct = sum(r["true_severity"]==r["pred_severity"] for r in ft_results)
print(f"\nFine-tuned severity accuracy: {sev_correct/len(ft_results):.1%} ({sev_correct}/{len(ft_results)})")

with open(os.path.join(ROOT,"Results","ft_results.json"),"w") as f:
    jlib.dump(ft_results, f)
print("Saved: ft_results.json")

In [ ]:
# Cell 7 -- FREE fine-tuned model, load baseline
# Critical: must free 6GB before loading baseline model
import gc

print("Freeing fine-tuned model from VRAM...")
del finetuned_model, base
gc.collect()
torch.cuda.empty_cache()

used  = torch.cuda.memory_allocated(0)/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"VRAM after free: {used:.1f}/{total:.1f} GB -- should be near 0")

# Load baseline (same model, NO adapter)
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print("\nLoading zero-shot baseline (no LoRA adapter)...")
bnb2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_processor  = AutoProcessor.from_pretrained(cfg["model"]["name"])
baseline_model  = AutoModelForImageTextToText.from_pretrained(
    cfg["model"]["name"],
    quantization_config=bnb2,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "7GiB", "cpu": "12GiB"},
)
baseline_model.eval()

used = torch.cuda.memory_allocated(0)/1e9
print(f"Baseline model ready | VRAM: {used:.1f}/{total:.1f} GB")

In [ ]:
# Cell 8 -- Zero-shot baseline evaluation (same test samples)
baseline_results = []
print(f"Evaluating zero-shot baseline on {len(test_sample)} samples...")

for _, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    symptom = SYMPTOM_LOOKUP.get(row["dx"], "Skin lesion noted.")
    result  = run_inference(baseline_model, base_processor, row["image_path"], symptom)
    baseline_results.append({
        "true_dx":       row["dx"],
        "true_severity": SEVERITY_MAP[row["dx"]],
        "pred_severity": result.get("severity","UNKNOWN").upper(),
    })

base_correct = sum(r["true_severity"]==r["pred_severity"] for r in baseline_results)
print(f"\nZero-shot severity accuracy: {base_correct/len(baseline_results):.1%}")

with open(os.path.join(ROOT,"Results","baseline_results.json"),"w") as f:
    jlib.dump(baseline_results, f)
print("Saved: baseline_results.json")

# Free baseline model too -- done with inference
del baseline_model
gc.collect(); torch.cuda.empty_cache()
print("VRAM freed.")

In [ ]:
# Cell 9 -- Comparison table + confusion matrices
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix
from collections import Counter

with open(os.path.join(ROOT,"Results","ft_results.json"))       as f: ft_results       = jlib.load(f)
with open(os.path.join(ROOT,"Results","baseline_results.json")) as f: baseline_results = jlib.load(f)

ft_acc   = sum(r["true_severity"]==r["pred_severity"] for r in ft_results)/len(ft_results)
base_acc = sum(r["true_severity"]==r["pred_severity"] for r in baseline_results)/len(baseline_results)

print("="*52)
print(f"  {'Model':<32} {'Severity Acc':>12}")
print("="*52)
print(f"  {'Zero-shot Qwen2.5-VL (no fine-tune)':<32} {base_acc:>11.1%}")
print(f"  {'RuralMed Vision (QLoRA fine-tuned)':<32} {ft_acc:>11.1%}")
print(f"  {'Improvement':<32} {ft_acc-base_acc:>+11.1%}")
print("="*52)

SEVS = ["LOW","MEDIUM","HIGH"]
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, results, title in [
    (axes[0], baseline_results, "Zero-Shot Baseline"),
    (axes[1], ft_results,       "RuralMed Vision (Fine-tuned)"),
]:
    true = [r["true_severity"] for r in results if r["true_severity"] in SEVS]
    pred = [r["pred_severity"] for r in results if r["true_severity"] in SEVS]
    cm   = confusion_matrix(true, pred, labels=SEVS)
    cm_n = cm.astype(float)
    rs   = cm_n.sum(axis=1, keepdims=True)
    cm_n = np.divide(cm_n, rs, where=rs!=0) * 100
    sns.heatmap(cm_n, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=SEVS, yticklabels=SEVS,
                linewidths=0.5, ax=ax, cbar_kws={"label":"%"})
    acc = sum(t==p for t,p in zip(true,pred))/max(len(true),1)
    ax.set_title(f"{title}\nSeverity Accuracy: {acc:.1%}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.suptitle("Severity Classification -- Baseline vs Fine-Tuned",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ROOT,"Results","figures","confusion_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_comparison.png")

In [ ]:
# Cell 10 -- Final summary figure (paper + professor meeting)

fig, axes = plt.subplots(1,3,figsize=(17,5))

# 1. Accuracy bar chart
axes[0].bar(["Zero-Shot\nBaseline","RuralMed Vision\n(Fine-tuned)"],
            [base_acc*100, ft_acc*100],
            color=["#95a5a6","#2980b9"], edgecolor="white", width=0.5)
axes[0].set_ylim(0,100)
axes[0].set_ylabel("Severity Accuracy (%)")
axes[0].set_title("Model Comparison", fontweight="bold")
for i,(acc) in enumerate([base_acc, ft_acc]):
    axes[0].text(i, acc*100+1, f"{acc:.1%}", ha="center", fontweight="bold", fontsize=13)
axes[0].axhline(33, color="red", linestyle="--", alpha=0.5, label="Random (3-class)")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2, axis="y")

# 2. Training curves
history_path = os.path.join(ROOT, cfg["model"]["adapter_path"], "training_history.json")
if os.path.isfile(history_path):
    with open(history_path) as f:
        history = jlib.load(f)
    st = [x["step"] for x in history if "train_loss" in x]
    lt = [x["train_loss"] for x in history if "train_loss" in x]
    se = [x["step"] for x in history if "eval_loss" in x]
    le = [x["eval_loss"] for x in history if "eval_loss" in x]
    axes[1].plot(st, lt, color="#1f77b4", linewidth=1.5, label="Train loss")
    if le:
        axes[1].plot(se, le, "o-", color="#d62728", linewidth=2, markersize=5, label="Eval loss")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Loss")
    axes[1].set_title("Training Convergence", fontweight="bold")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5,0.5,"training_history.json\nnot found",
                 ha="center",va="center",transform=axes[1].transAxes)

# 3. Severity distribution
SEVS = ["LOW","MEDIUM","HIGH"]
gt_dist   = Counter(r["true_severity"] for r in ft_results)
pred_dist = Counter(r["pred_severity"] for r in ft_results)
x = np.arange(3); w = 0.35
axes[2].bar(x-w/2,[gt_dist.get(s,0) for s in SEVS],w,
            label="Ground truth",color="#2ecc71",edgecolor="white")
axes[2].bar(x+w/2,[pred_dist.get(s,0) for s in SEVS],w,
            label="Predicted",color="#3498db",edgecolor="white")
axes[2].set_xticks(x); axes[2].set_xticklabels(SEVS)
axes[2].set_ylabel("Sample count")
axes[2].set_title("Severity Distribution: GT vs Predicted", fontweight="bold")
axes[2].legend(); axes[2].grid(True, alpha=0.2, axis="y")

plt.suptitle("RuralMed Vision -- Evaluation Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ROOT,"Results","figures","evaluation_summary.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: evaluation_summary.png")
print(f"\nKey numbers for paper/meeting:")
print(f"  Zero-shot accuracy : {base_acc:.1%}")
print(f"  Fine-tuned accuracy: {ft_acc:.1%}")
print(f"  Improvement        : {ft_acc-base_acc:+.1%}")

---
## Section 2 -- Statistical Metrics & Ablations

Adds the following for the paper:
- Bootstrap confidence intervals on accuracy
- Per-class precision, recall, F1, MCC, Cohen's Kappa
- RMSE and MAE on severity score predictions
- Ablation: text-only vs multimodal (image+text)
- Extended XAI: per-class Grad-CAM statistics

In [ ]:
# Cell 11 -- Load results + full per-class statistical metrics
import json as jlib, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef, precision_recall_fscore_support)
from scipy import stats

with open(os.path.join(ROOT,"Results","ft_results.json"))       as f: ft_results       = jlib.load(f)
with open(os.path.join(ROOT,"Results","baseline_results.json")) as f: baseline_results = jlib.load(f)

SEVS      = ["LOW","MEDIUM","HIGH"]
SEV_NUM   = {"LOW":0,"MEDIUM":1,"HIGH":2}   # for RMSE/MAE

ft_true  = [r["true_severity"] for r in ft_results   if r["true_severity"] in SEVS]
ft_pred  = [r["pred_severity"] for r in ft_results   if r["true_severity"] in SEVS]
b_true   = [r["true_severity"] for r in baseline_results if r["true_severity"] in SEVS]
b_pred   = [r["pred_severity"] for r in baseline_results if r["true_severity"] in SEVS]

# ── Bootstrap confidence interval ────────────────────────────────────────────
def bootstrap_ci(y_true, y_pred, n_boot=1000, ci=0.95):
    accs = []
    n    = len(y_true)
    rng  = np.random.default_rng(42)
    for _ in range(n_boot):
        idx  = rng.integers(0, n, size=n)
        acc  = sum(y_true[i]==y_pred[i] for i in idx) / n
        accs.append(acc)
    lo = np.percentile(accs, (1-ci)/2*100)
    hi = np.percentile(accs, (1-(1-ci)/2)*100)
    return lo, hi

ft_acc  = sum(t==p for t,p in zip(ft_true,ft_pred))  / len(ft_true)
b_acc   = sum(t==p for t,p in zip(b_true,b_pred))    / len(b_true)
ft_lo,  ft_hi  = bootstrap_ci(ft_true,  ft_pred)
b_lo,   b_hi   = bootstrap_ci(b_true,   b_pred)

# ── Cohen's Kappa + MCC ──────────────────────────────────────────────────────
ft_kappa = cohen_kappa_score(ft_true, ft_pred, labels=SEVS)
b_kappa  = cohen_kappa_score(b_true,  b_pred,  labels=SEVS)
ft_mcc   = matthews_corrcoef(ft_true, ft_pred)
b_mcc    = matthews_corrcoef(b_true,  b_pred)

# ── RMSE / MAE on ordinal severity ───────────────────────────────────────────
ft_true_n  = [SEV_NUM[s] for s in ft_true]
ft_pred_n  = [SEV_NUM.get(s,1) for s in ft_pred]
b_true_n   = [SEV_NUM[s] for s in b_true]
b_pred_n   = [SEV_NUM.get(s,1) for s in b_pred]

ft_rmse = np.sqrt(np.mean((np.array(ft_true_n)-np.array(ft_pred_n))**2))
b_rmse  = np.sqrt(np.mean((np.array(b_true_n) -np.array(b_pred_n)) **2))
ft_mae  = np.mean(np.abs(np.array(ft_true_n)-np.array(ft_pred_n)))
b_mae   = np.mean(np.abs(np.array(b_true_n) -np.array(b_pred_n)))

# ── Per-class P/R/F1 ─────────────────────────────────────────────────────────
ft_p, ft_r, ft_f, _ = precision_recall_fscore_support(ft_true, ft_pred, labels=SEVS, zero_division=0)
b_p,  b_r,  b_f,  _ = precision_recall_fscore_support(b_true,  b_pred,  labels=SEVS, zero_division=0)

# ── Print full table ──────────────────────────────────────────────────────────
print("="*65)
print(f"  {'Metric':<35} {'Baseline':>12} {'Fine-tuned':>12}")
print("="*65)
print(f"  {'Severity Accuracy':<35} {b_acc:>11.1%} {ft_acc:>11.1%}")
print(f"  {'95% CI':<35} [{b_lo:.1%},{b_hi:.1%}]  [{ft_lo:.1%},{ft_hi:.1%}]")
print(f"  {'Cohen Kappa':<35} {b_kappa:>11.3f} {ft_kappa:>11.3f}")
print(f"  {'Matthews Corr Coef (MCC)':<35} {b_mcc:>11.3f} {ft_mcc:>11.3f}")
print(f"  {'RMSE (ordinal severity)':<35} {b_rmse:>11.3f} {ft_rmse:>11.3f}")
print(f"  {'MAE  (ordinal severity)':<35} {b_mae:>11.3f} {ft_mae:>11.3f}")
print("="*65)
print()
print(f"  Per-class metrics (Fine-tuned):")
print(f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*42}")
for cls, p, r, f in zip(SEVS, ft_p, ft_r, ft_f):
    print(f"  {cls:<12} {p:>10.3f} {r:>10.3f} {f:>10.3f}")
print()
print(f"  Per-class metrics (Zero-shot Baseline):")
print(f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*42}")
for cls, p, r, f in zip(SEVS, b_p, b_r, b_f):
    print(f"  {cls:<12} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

# Store for later cells
metrics = {
    "ft_acc": float(ft_acc), "ft_lo": float(ft_lo), "ft_hi": float(ft_hi),
    "ft_kappa": float(ft_kappa), "ft_mcc": float(ft_mcc),
    "ft_rmse": float(ft_rmse), "ft_mae": float(ft_mae),
    "ft_p": [float(x) for x in ft_p],
    "ft_r": [float(x) for x in ft_r],
    "ft_f": [float(x) for x in ft_f],
    "b_acc": float(b_acc), "b_lo": float(b_lo), "b_hi": float(b_hi),
    "b_kappa": float(b_kappa), "b_mcc": float(b_mcc),
    "b_rmse": float(b_rmse), "b_mae": float(b_mae),
    "b_p": [float(x) for x in b_p],
    "b_r": [float(x) for x in b_r],
    "b_f": [float(x) for x in b_f],
}
with open(os.path.join(ROOT,"Results","metrics.json"),"w") as f: jlib.dump(metrics, f, indent=2)
print("Saved: metrics.json")
print(f"Keys: {list(metrics.keys())}")

In [ ]:
# Cell 12 -- Comprehensive metrics figure (paper-ready)
import numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix
import seaborn as sns

with open(os.path.join(ROOT,"Results","metrics.json")) as f: m = jlib.load(f)

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Accuracy + CI ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
models  = ["Zero-Shot\nBaseline", "RuralMed\nVision"]
accs    = [m["b_acc"]*100, m["ft_acc"]*100]
errs_lo = [accs[0]-m["b_lo"]*100, accs[1]-m["ft_lo"]*100]
errs_hi = [m["b_hi"]*100-accs[0], m["ft_hi"]*100-accs[1]]
colors  = ["#95a5a6","#2980b9"]
bars = ax1.bar(models, accs, color=colors, edgecolor="white", width=0.5,
               yerr=[errs_lo, errs_hi], capsize=8, error_kw={"elinewidth":2})
ax1.set_ylim(0,100)
ax1.set_ylabel("Severity Accuracy (%)")
ax1.set_title("Accuracy + 95% CI\n(Bootstrap, n=1000)", fontweight="bold")
for bar,acc in zip(bars,accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
             f"{acc:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax1.axhline(33, color="red", linestyle="--", alpha=0.5, label="Random")
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.2, axis="y")

# ── 2. Kappa + MCC ───────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
metric_names = ["Cohen's\nKappa", "Matthews\nMCC"]
b_vals  = [m["b_kappa"],  m["b_mcc"]]
ft_vals = [m["ft_kappa"], m["ft_mcc"]]
x = np.arange(2); w = 0.3
ax2.bar(x-w/2, b_vals,  w, label="Baseline",    color="#95a5a6", edgecolor="white")
ax2.bar(x+w/2, ft_vals, w, label="Fine-tuned",  color="#2980b9", edgecolor="white")
ax2.set_xticks(x); ax2.set_xticklabels(metric_names)
ax2.set_title("Agreement Metrics\n(1.0 = perfect)", fontweight="bold")
ax2.axhline(0, color="black", linewidth=0.8, alpha=0.5)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.2, axis="y")
for i,(b,ft) in enumerate(zip(b_vals,ft_vals)):
    ax2.text(i-w/2, b+0.02, f"{b:.3f}", ha="center", fontsize=8)
    ax2.text(i+w/2, ft+0.02, f"{ft:.3f}", ha="center", fontsize=8, fontweight="bold")

# ── 3. RMSE / MAE ────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
err_names = ["RMSE", "MAE"]
b_errs  = [m["b_rmse"],  m["b_mae"]]
ft_errs = [m["ft_rmse"], m["ft_mae"]]
x = np.arange(2)
ax3.bar(x-w/2, b_errs,  w, label="Baseline",   color="#e74c3c", edgecolor="white", alpha=0.8)
ax3.bar(x+w/2, ft_errs, w, label="Fine-tuned", color="#27ae60", edgecolor="white", alpha=0.8)
ax3.set_xticks(x); ax3.set_xticklabels(err_names)
ax3.set_title("Ordinal Severity Error\n(lower is better)", fontweight="bold")
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.2, axis="y")
for i,(b,ft) in enumerate(zip(b_errs,ft_errs)):
    ax3.text(i-w/2, b+0.01, f"{b:.3f}", ha="center", fontsize=8)
    ax3.text(i+w/2, ft+0.01, f"{ft:.3f}", ha="center", fontsize=8, fontweight="bold")

# ── 4. Per-class F1 comparison ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
x = np.arange(3); w = 0.3
ax4.bar(x-w/2, m["b_f"],  w, label="Baseline",   color="#95a5a6", edgecolor="white")
ax4.bar(x+w/2, m["ft_f"], w, label="Fine-tuned", color="#2980b9", edgecolor="white")
ax4.set_xticks(x); ax4.set_xticklabels(["LOW","MEDIUM","HIGH"])
ax4.set_ylabel("F1 Score")
ax4.set_title("Per-Class F1 Score: Baseline vs Fine-Tuned", fontweight="bold")
ax4.set_ylim(0,1)
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.2, axis="y")
for i,(b,ft) in enumerate(zip(m["b_f"],m["ft_f"])):
    ax4.text(i-w/2, b+0.01, f"{b:.2f}", ha="center", fontsize=9)
    ax4.text(i+w/2, ft+0.01, f"{ft:.2f}", ha="center", fontsize=9, fontweight="bold")

# ── 5. Precision vs Recall scatter ───────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
cls_colors = {"LOW":"#27ae60","MEDIUM":"#e67e22","HIGH":"#c0392b"}
for cls, p_b, r_b, p_ft, r_ft in zip(SEVS, m["b_p"], m["b_r"], m["ft_p"], m["ft_r"]):
    c = cls_colors[cls]
    ax5.scatter(r_b,  p_b,  c=c, marker="x", s=120, linewidths=2, label=f"{cls} (base)")
    ax5.scatter(r_ft, p_ft, c=c, marker="o", s=120, label=f"{cls} (ft)")
    ax5.annotate(f"{cls}", (r_ft, p_ft), textcoords="offset points",
                 xytext=(5,5), fontsize=8)
ax5.plot([0,1],[0,1],"k--",alpha=0.3)
ax5.set_xlabel("Recall"); ax5.set_ylabel("Precision")
ax5.set_title("Precision-Recall\n(x=baseline, o=fine-tuned)", fontweight="bold")
ax5.set_xlim(0,1.05); ax5.set_ylim(0,1.05)
ax5.grid(True, alpha=0.3)

# ── 6. Summary table as heatmap ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :])
table_data = np.array([
    [m["b_acc"],  m["b_kappa"], m["b_mcc"],  m["b_rmse"],  m["b_mae"],
     m["b_p"][0], m["b_r"][0], m["b_f"][0],
     m["b_p"][2], m["b_r"][2], m["b_f"][2]],
    [m["ft_acc"], m["ft_kappa"],m["ft_mcc"], m["ft_rmse"], m["ft_mae"],
     m["ft_p"][0],m["ft_r"][0],m["ft_f"][0],
     m["ft_p"][2],m["ft_r"][2],m["ft_f"][2]],
])
cols = ["Accuracy","Kappa","MCC","RMSE","MAE",
        "P(LOW)","R(LOW)","F1(LOW)",
        "P(HIGH)","R(HIGH)","F1(HIGH)"]
rows = ["Zero-Shot\nBaseline","RuralMed\nVision"]
sns.heatmap(table_data, annot=True, fmt=".3f", cmap="RdYlGn",
            xticklabels=cols, yticklabels=rows,
            linewidths=0.5, ax=ax6,
            cbar_kws={"label":"Score (higher=better except RMSE/MAE)"})
ax6.set_title("Full Metrics Summary Table", fontweight="bold", pad=10)
ax6.tick_params(axis="x", rotation=30)

fig.suptitle("RuralMed Vision -- Comprehensive Evaluation Metrics", fontsize=15, fontweight="bold")
plt.savefig(os.path.join(ROOT,"Results","figures","metrics_comprehensive.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: metrics_comprehensive.png")

In [ ]:
# Cell 13 -- Ablation study: text-only vs multimodal (image+text)
# Shows the contribution of the image modality to triage accuracy
# This is a key ablation for the paper

import torch, gc, json as jlib, re
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# Reload fine-tuned model for ablation
print("Reloading fine-tuned model for ablation study...")
torch.cuda.empty_cache(); gc.collect()

ADAPTER_PATH = cfg["model"]["adapter_path"]
processor_ab = AutoProcessor.from_pretrained(ADAPTER_PATH)
bnb_ab = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base_ab = AutoModelForImageTextToText.from_pretrained(
    cfg["model"]["name"], quantization_config=bnb_ab,
    device_map="auto", torch_dtype=torch.bfloat16,
    max_memory={0: "7GiB", "cpu": "12GiB"})
ablation_model = PeftModel.from_pretrained(base_ab, ADAPTER_PATH)
ablation_model.eval()
print("Model ready for ablation")

ABLATION_SYSTEM = """You are a medical triage assistant. Given skin lesion symptoms,
classify severity. Respond with ONLY JSON: {"severity": "<LOW, MEDIUM, or HIGH>"}"""

def run_text_only(model, proc, symptom_text):
    """Run inference with text ONLY -- no image input"""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": ABLATION_SYSTEM}]},
        {"role": "user",   "content": [{"type": "text",
                                        "text": f"Symptoms: {symptom_text}\nClassify severity."}]},
    ]
    text   = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=text, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False,
                             pad_token_id=proc.tokenizer.eos_token_id)
    raw = proc.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        parsed = jlib.loads(re.sub(r"```json|```","",raw).strip())
        return parsed.get("severity","UNKNOWN").upper()
    except:
        return next((s for s in ["HIGH","MEDIUM","LOW"] if s in raw.upper()), "UNKNOWN")

# Run on same test_sample used for evaluation
ABLATION_N = min(100, len(test_sample))
ablation_sample = test_sample.sample(n=ABLATION_N, random_state=42).reset_index(drop=True)

text_only_results = []
print(f"Running text-only ablation on {ABLATION_N} samples...")
for _, row in tqdm(ablation_sample.iterrows(), total=ABLATION_N):
    symptom = SYMPTOM_LOOKUP.get(row["dx"], "Skin lesion noted.")
    pred    = run_text_only(ablation_model, processor_ab, symptom)
    text_only_results.append({
        "true_severity": SEVERITY_MAP[row["dx"]],
        "pred_severity": pred,
    })

text_acc = sum(r["true_severity"]==r["pred_severity"] for r in text_only_results)/len(text_only_results)

# Compare: text-only vs multimodal on same subset
ft_subset = [ft_results[i] for i in range(min(ABLATION_N, len(ft_results)))]
mm_acc    = sum(r["true_severity"]==r["pred_severity"] for r in ft_subset)/len(ft_subset)

print(f"\nAblation Results ({ABLATION_N} samples):")
print(f"  Text-only (no image)     : {text_acc:.1%}")
print(f"  Multimodal (image+text)  : {mm_acc:.1%}")
print(f"  Image contribution       : {mm_acc-text_acc:+.1%}")

with open(os.path.join(ROOT,"Results","ablation_results.json"),"w") as f:
    jlib.dump({"text_only": text_only_results, "text_acc": text_acc,
               "mm_acc": mm_acc, "n": ABLATION_N}, f)

del ablation_model, base_ab
gc.collect(); torch.cuda.empty_cache()
print("VRAM freed.")

In [ ]:
# Cell 14 -- Ablation figure + clinical error analysis
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import json as jlib
from sklearn.metrics import confusion_matrix

with open(os.path.join(ROOT,"Results","ablation_results.json")) as f: abl = jlib.load(f)

text_acc = abl["text_acc"]
mm_acc   = abl["mm_acc"]

with open(os.path.join(ROOT,"Results","baseline_results.json")) as f: baseline_results = jlib.load(f)
with open(os.path.join(ROOT,"Results","metrics.json"))          as f: m                = jlib.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Ablation bar chart
ax = axes[0]
labels = ["Zero-Shot\n(no fine-tune)", "Fine-tuned\nText-Only", "Fine-tuned\nMultimodal"]
accs   = [m["b_acc"], text_acc, mm_acc]
colors = ["#95a5a6","#f39c12","#2980b9"]
bars   = ax.bar(labels, [a*100 for a in accs], color=colors, edgecolor="white", width=0.55)
ax.set_ylim(0,100)
ax.set_ylabel("Severity Accuracy (%)")
ax.set_title("Ablation Study\nZero-shot vs Text-only vs Multimodal", fontweight="bold")
for bar,acc in zip(bars,accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f"{acc:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax.axhline(33, color="red", linestyle="--", alpha=0.4, label="Random")
ax.legend(fontsize=9); ax.grid(True, alpha=0.2, axis="y")

# Annotate improvement arrows
for i in range(1, len(accs)):
    diff = accs[i] - accs[i-1]
    ax.annotate(f"{diff:+.1f}%", xy=(i, accs[i]*100-5), ha="center",
                fontsize=9, color="#27ae60" if diff>0 else "#e74c3c", fontweight="bold")

# 2. Clinical error analysis -- HIGH severity misclassified as LOW (dangerous)
ax2 = axes[1]
SEVS = ["LOW","MEDIUM","HIGH"]

with open(os.path.join(ROOT,"Results","ft_results.json")) as f: ft_results = jlib.load(f)

# Critical errors: HIGH predicted as LOW (missed dangerous case)
ft_true = [r["true_severity"] for r in ft_results if r["true_severity"] in SEVS]
ft_pred = [r["pred_severity"] for r in ft_results if r["true_severity"] in SEVS]
b_true  = [r["true_severity"] for r in baseline_results if r["true_severity"] in SEVS]
b_pred  = [r["pred_severity"] for r in baseline_results if r["true_severity"] in SEVS]

def critical_errors(true, pred):
    # HIGH -> LOW: most dangerous (missed melanoma)
    h_to_l = sum(1 for t,p in zip(true,pred) if t=="HIGH" and p=="LOW")
    # HIGH -> MEDIUM: dangerous (downgraded)
    h_to_m = sum(1 for t,p in zip(true,pred) if t=="HIGH" and p=="MEDIUM")
    # LOW -> HIGH: over-triage (unnecessary)
    l_to_h = sum(1 for t,p in zip(true,pred) if t=="LOW"  and p=="HIGH")
    total_high = sum(1 for t in true if t=="HIGH")
    return h_to_l, h_to_m, l_to_h, total_high

b_h2l,  b_h2m,  b_l2h,  b_tot  = critical_errors(b_true,  b_pred)
ft_h2l, ft_h2m, ft_l2h, ft_tot = critical_errors(ft_true, ft_pred)

error_types = ["HIGH\nmissed as LOW\n(dangerous)", "HIGH\ndowngraded\nto MEDIUM", "LOW\nover-triaged\nto HIGH"]
b_errs  = [b_h2l/max(b_tot,1),   b_h2m/max(b_tot,1),   b_l2h/max(len(b_true),1)]
ft_errs = [ft_h2l/max(ft_tot,1), ft_h2m/max(ft_tot,1), ft_l2h/max(len(ft_true),1)]
x = np.arange(3); w=0.3
ax2.bar(x-w/2, [e*100 for e in b_errs],  w, label="Baseline",   color="#e74c3c", alpha=0.85, edgecolor="white")
ax2.bar(x+w/2, [e*100 for e in ft_errs], w, label="Fine-tuned", color="#27ae60", alpha=0.85, edgecolor="white")
ax2.set_xticks(x); ax2.set_xticklabels(error_types, fontsize=9)
ax2.set_ylabel("Error Rate (% of category)")
ax2.set_title("Clinical Error Analysis\n(lower is better)", fontweight="bold")
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.2, axis="y")

# 3. Severity confusion matrix difference (improvement heatmap)
cm_b  = confusion_matrix(b_true,  b_pred,  labels=SEVS).astype(float)
cm_ft = confusion_matrix(ft_true, ft_pred, labels=SEVS).astype(float)
rs_b  = cm_b.sum(axis=1,  keepdims=True)
rs_ft = cm_ft.sum(axis=1, keepdims=True)
cm_b_n  = np.divide(cm_b,  rs_b,  where=rs_b!=0)  * 100
cm_ft_n = np.divide(cm_ft, rs_ft, where=rs_ft!=0) * 100
diff    = cm_ft_n - cm_b_n

sns.heatmap(diff, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            xticklabels=SEVS, yticklabels=SEVS, linewidths=0.5, ax=axes[2],
            cbar_kws={"label":"Fine-tuned improvement (% points)"})
axes[2].set_title("Confusion Matrix Improvement\n(fine-tuned minus baseline)", fontweight="bold")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("Actual")

plt.suptitle("RuralMed Vision -- Ablation Study & Clinical Error Analysis",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ROOT,"Results","figures","ablation_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Critical error reduction:")
print(f"  HIGH missed as LOW : {b_h2l} -> {ft_h2l} cases ({b_h2l-ft_h2l:+d})")
print(f"  HIGH downgraded    : {b_h2m} -> {ft_h2m} cases ({b_h2m-ft_h2m:+d})")
print("Saved: ablation_analysis.png")

In [ ]:
# Cell 15 -- Paper-ready numbers summary (copy these into your paper)
import json as jlib
with open(os.path.join(ROOT,"Results","metrics.json")) as f: m = jlib.load(f)
with open(os.path.join(ROOT,"Results","ablation_results.json")) as f: abl = jlib.load(f)

print("="*60)
print("PAPER-READY NUMBERS -- copy these into your manuscript")
print("="*60)
print()
print("Table 1: Main Results")
print(f"  Zero-shot baseline accuracy : {m['b_acc']:.1%} (95% CI: {m['b_lo']:.1%}-{m['b_hi']:.1%})")
print(f"  RuralMed Vision accuracy    : {m['ft_acc']:.1%} (95% CI: {m['ft_lo']:.1%}-{m['ft_hi']:.1%})")
print(f"  Absolute improvement        : {m['ft_acc']-m['b_acc']:+.1%}")
print(f"  Cohen's Kappa (fine-tuned)  : {m['ft_kappa']:.3f}")
print(f"  MCC (fine-tuned)            : {m['ft_mcc']:.3f}")
print(f"  RMSE severity (fine-tuned)  : {m['ft_rmse']:.3f}")
print(f"  MAE  severity (fine-tuned)  : {m['ft_mae']:.3f}")
print()
print("Table 2: Per-Class F1 (Fine-tuned)")
for cls, f1 in zip(["LOW","MEDIUM","HIGH"], m["ft_f"]):
    print(f"  {cls:<10}: F1 = {f1:.3f}")
print()
print("Table 3: Ablation Study")
print(f"  Zero-shot (no fine-tune)    : {m['b_acc']:.1%}")
print(f"  Fine-tuned, text-only       : {abl['text_acc']:.1%}")
print(f"  Fine-tuned, multimodal      : {abl['mm_acc']:.1%}")
print(f"  Image modality contribution : {abl['mm_acc']-abl['text_acc']:+.1%}")
print()
print("Generated figures for paper:")
print("  metrics_comprehensive.png  -- main metrics table + per-class analysis")
print("  ablation_analysis.png      -- ablation + clinical error analysis")
print("  confusion_comparison.png   -- confusion matrices baseline vs fine-tuned")
print("  evaluation_summary.png     -- overview figure")
print("  training_curves.png        -- loss + perplexity convergence")
print("  triage_gallery.png         -- qualitative results with Grad-CAM")
print("  calibration.png            -- MC Dropout reliability diagram")
print()
print("One-line result for abstract:")
print(f"  'RuralMed Vision achieves {m['ft_acc']:.0%} severity classification accuracy")
print(f"   on HAM10000 (kappa={m['ft_kappa']:.2f}), compared to {m['b_acc']:.0%} zero-shot,")
print(f"   while running offline on an 8GB consumer GPU.'")